<a href="https://colab.research.google.com/github/HowardChen0211/dataManPy/blob/main/ProblemSet_3_4_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# California Housing Market, 2020-2025: Demand, Affordability, and Mortgage Rates

### Research questions
1. Which California counties had the biggest increase in home values from January 2020 to December 2025?

2. Is there a relationship between home-value growth and rent growth across counties?

3. How did home-value growth change when the 30-year mortgage rate increased from below 5% to above 5%?

4. Are population size and unemployment related to home-value growth?

5. Did counties with faster population growth also have faster home-value growth?

6. Which California counties have the least affordable homes compared with household income?


### Why these data
I use Zillow housing data together with county-level population and labor-market data. Zillow provides data on home values and rents, while the other data provide information about each county.

For PS3, I also add the 30-year mortgage rate from Freddie Mac, median household income and home value from the ACS, and the 2020 population.

Together, these datasets help me understand how housing markets are different across California counties and how housing prices are related to population and economic factors.

### Unit of analysis
California counties (n = 58). County-level datasets are merged on the FIPS code, and the national mortgage rate is merged on month.

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#will display all output not just last command
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import plotly.express as px #a quick simple one
import plotly.graph_objects as go #can get convoluted

pd.set_option('display.float_format', '{:.3f}'.format) # Turn off the scientific notation https://medium.com/@amit25173/steps-to-pandas-turn-off-scientific-notation-62c44b86ee24

# ProblemSet 0

## Load the data

In [ ]:
# Zillow Home Value Index (ZHVI)
zhvi = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")

# Zillow Observed Rent Index (ZORI)
zori = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/County_zori_uc_sfrcondomfr_sm_month.csv")

# County population 2025
pop_2025 = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/CA_Population_estimates_2025.csv")

# ACS 2020-2024 5-year: civilian labor force 16 and over,
unemployment = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/CA_UnemploymentRate.csv")

In [ ]:
# take a look about the data (first 5 rows)
zhvi.head()

In [ ]:
# take a look about the data (first 5 rows)
zori.head()

In [ ]:
# take a look about the data (first 5 rows)
pop_2025.head()

In [ ]:
# take a look about the data (first 5 rows)
unemployment.head()

**Interpretation**: Both Zillow files are national and wide, with one row per U.S. county and one column per month, so I have to filter to California and reshape them before doing anything else.

# ProblemSet 1

#### Data Inspection

In [ ]:
# prints information about a DataFrame
print(zhvi.info(), "\n")
print(zori.info(), "\n")
print(pop_2025.info(), "\n")
print(unemployment.info(), "\n")

In [ ]:
# See how many columns in the data
print("Home Index: ", len(zhvi.columns))
print("Rent Index: ", len(zori.columns))
print("Population: ", len(pop_2025.columns))
print("Unemployment: ", len(unemployment.columns))

In [ ]:
# See how big the data
print(zhvi.shape) # (3071, 329)
print(zori.shape) # (1407, 149)
print(pop_2025.shape) # (59, 5)
print(unemployment.shape) # (59, 9)

In [ ]:
# Missing values
print(zhvi.isna().sum().sum())
print()
print(zori.isna().sum().sum())
print()
print(pop_2025.isna().sum().sum())
print()
print(unemployment.isna().sum().sum())

In [ ]:
# Check Duplicates
print(zhvi.duplicated().sum())
print(zori.duplicated().sum())
print(pop_2025.duplicated().sum())
print(unemployment.duplicated().sum())

**Interpretation**: The population and unemployment files have 59 rows, which is the 58 counties plus one code row at the top. Both Zillow files have many missing values, mostly because some counties start later than others. None of the four files has duplicate rows.

#### Data Cleaning

Transform both zillow dataset into long format *ZHVI* & *ZORI* and filtered the dates to include only records from 2020 to 2025.

In [ ]:
START, END = "2020-01", "2026-01"

def zillow_to_long(data, value_name, start=START, end=END):

    cols = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "StateCodeFIPS", "MunicipalCodeFIPS"]
    dates = [i for i in data.columns if i[0].isdigit() and start <= i < end]

    ca = data.loc[data["StateName"] == "CA", cols + dates].copy()
    # FIPS codes is 00+000
    ca["FIPS"] = (ca["StateCodeFIPS"].astype(str).str.zfill(2)
                  + ca["MunicipalCodeFIPS"].astype(str).str.zfill(3))

    ca["RegionName"] = ca["RegionName"].str.strip()

    # wide to long https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    df = ca.melt(id_vars=["FIPS", "RegionName"], value_vars=dates,
                   var_name="Date", value_name=value_name)

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    return df.sort_values(["FIPS", "Date"]).reset_index(drop=True)

# Call the function
zhvi_CA = zillow_to_long(zhvi, "ZHVI")
zori_CA = zillow_to_long(zori, "ZORI")

# See the shape & check unique values
print("ZHVI:", zhvi_CA.shape, "| counties:", zhvi_CA["FIPS"].nunique(), "| months:", zhvi_CA["Date"].nunique())
print("ZORI:", zori_CA.shape, "| counties:", zori_CA["FIPS"].nunique(), "| months:", zori_CA["Date"].nunique())

ZHVI covers all 58 counties × 72 months = 4,176 rows. ZORI has fewer counties because Zillow does not publish rents for some small rural counties.

In [ ]:
# See the data types in the dataframe
zhvi_CA.dtypes

In [ ]:
zori_CA.dtypes

In [ ]:
zhvi_CA.head()

In [ ]:
zori_CA.head()

### Data Manipulation

#### Merge 1: ZHVI x ZORI housing

In [ ]:
housing = zhvi_CA.merge(
    zori_CA.drop(columns="RegionName"),
    on=["FIPS", "Date"],
    how="outer",            # keep every county-month that has a home value
    validate="1:1",
    indicator=True,
)

In [ ]:
housing.query("_merge == 'left_only'")

In [ ]:
# checks
print(housing["_merge"].value_counts(), "\n")

In [ ]:
no_rent = sorted(housing.loc[housing["_merge"] == "left_only", "RegionName"].unique())
print(f"{len(no_rent)} counties have no ZORI in any month:\n", no_rent)
housing = housing.drop(columns="_merge")

**Interpretation**: The outer join keeps all 4,176 county-months (58 x 72), so no home-value observation is lost. The 648 `left_only` rows are the 9 counties Zillow does not publish rents for at all (Alpine, Colusa, Lassen, Mariposa, Modoc, Mono, Plumas, Sierra, Trinity): small rural counties with few rental listings for a stable index. That is expected and is not a merge error.

#### Merge 2: Unemployment x Population

#### Population (from Social Explorer)

In [ ]:
pop_2025.head()

In [ ]:
pop_2025.info()

In [ ]:
pop_2025 = (pop_2025.iloc[1:,:][["FIPS Code", "Area Name", "Total Population (July 1, 2025)"]].rename(columns={"FIPS Code": "FIPS", "Area Name": "County", "Total Population (July 1, 2025)": "pop_2025"}))

pop_2025["pop_2025"] = pop_2025["pop_2025"].astype(int)

print(pop_2025.shape)
pop_2025.head()

#### Unemployment (ACS 2020–2024 5-year Estimates)

In [ ]:
# row 0 holds the (Geo_FIPS, SE_A17005_001, ...), so drop it
unemployment = (unemployment
                .loc[1:, ["FIPS", "NAME",
                          "Civilian Population in Labor Force 16 Years and Over",
                          "Civilian Population in Labor Force 16 Years and Over: Unemployed"]]
                .rename(columns={"NAME": "County",
                                 "Civilian Population in Labor Force 16 Years and Over": "labor_force",
                                 "Civilian Population in Labor Force 16 Years and Over: Unemployed": "unemployed"}))

unemployment["County"] = unemployment["County"].str.replace(", California", "")

unemployment["FIPS"] = unemployment["FIPS"].astype(str).str.zfill(5)

unemployment[["labor_force", "unemployed"]] = unemployment[["labor_force", "unemployed"]].astype(int)

unemployment["unemp_rate"] = unemployment["unemployed"] / unemployment["labor_force"] * 100

print(unemployment.shape)

unemployment.head(3)

In [ ]:
labor = unemployment.merge(pop_2025.drop(columns="County"), on="FIPS",
                           how="outer", validate="1:1", indicator=True)

print(labor["_merge"].value_counts())

labor = labor.drop(columns="_merge")

labor.head()

**Interpretation**: All 58 counties matched on both sides, with no `left_only` or `right_only` rows. That's what I expected, since both files are Social Explorer exports for the same list of California counties and FIPS is kept as a 5-digit string in both.

In [ ]:
labor.info()

## Mapping: 1. Counties to Regions
Group into Six Regions

Source: https://coolcalifornia.arb.ca.gov/regions

In [ ]:
regions = {
    "Bay Area": ["Alameda", "Contra Costa", "Marin", "Napa", "San Francisco",
                 "San Mateo", "Santa Clara", "Solano", "Sonoma"],
    "Southern California": ["Imperial", "Los Angeles", "Orange", "Riverside",
                            "San Bernardino", "San Diego", "Ventura"],
    "Central Coast": ["Monterey", "San Benito", "San Luis Obispo", "Santa Barbara", "Santa Cruz"],
    "Central Valley": ["Butte", "Colusa", "Fresno", "Glenn", "Kern", "Kings", "Madera",
                       "Merced", "Sacramento", "San Joaquin", "Stanislaus", "Sutter",
                       "Tehama", "Tulare", "Yolo", "Yuba"],
    "Sierra / Gold Country": ["Alpine", "Amador", "Calaveras", "El Dorado", "Inyo",
                              "Mariposa", "Mono", "Nevada", "Placer", "Plumas",
                              "Sierra", "Tuolumne"],
    "North Coast / Far North": ["Del Norte", "Humboldt", "Lake", "Lassen", "Mendocino",
                                "Modoc", "Shasta", "Siskiyou", "Trinity"],
}

county_to_region = {}

for region, county in regions.items():
  for c in county:
    county_to_region[f"{c} County"] = region

housing["region"] = housing["RegionName"].map(county_to_region)

housing.drop_duplicates("FIPS")["region"].value_counts() # Total should be 58

**Interpretation**: Every county got a region, since the counts add up to 58 and nothing was left. The groups are pretty uneven, the Central Valley has 16 counties and the Central Coast only 5, so the averages for the smaller regions rest on just a handful of places.

## Mapping 2: population to county size

In [ ]:
labor["size_tier"] = pd.cut(labor["pop_2025"],
                            bins=[0, 100_000, 1_000_000, np.inf],
                            labels=["Small (<100k)", "Mid (100k-1M)", "Large (1M+)"])
labor["size_tier"].value_counts().sort_index()

**Interpretation**: Most California counties are small or mid-sized, and only 10 of the 58 have more than a million people. The 100k and 1M cutoffs are my own round numbers rather than an official definition, but they're easy to explain and leave each group with a reasonable number of counties.

# ProblemSet 2

#### Groupby/Agg 1: County-level housing growth

I first aggregate the monthly Zillow data to the county level. This allows me to compare home values and rents at the beginning.

**!!! Note: I filtered the county that only has the complete 72 months of ZORI data, so that I can compare !!!!**

In [ ]:
housing_full_zori_county = housing[~housing["ZORI"].isna()]

housing_full_zori_county = housing_full_zori_county.groupby(["FIPS", "RegionName"]).agg(months=("Date", "nunique"))

# This line of code I'm using Gemini to help me how to get the values from index
housing_full_zori_county = housing_full_zori_county[housing_full_zori_county["months"] == 72].index.get_level_values('RegionName').unique().to_list()

In [ ]:
len(housing_full_zori_county)

**Interpretation**: Only 29 of the 58 counties have rent data for all 72 months, so the rent-versus-home-value comparison is based on these 29. Most of the dropped counties are small rural ones whose ZORI series starts late or doesn't exist at all.

In [ ]:
housings = housing[housing["RegionName"].isin(housing_full_zori_county)].reset_index(drop=True)

##### RQ1: Which California counties saw the fastest growth in home values from 2020 to 2025?

In [ ]:
# Group monthly housing data by county
county_housing = (
    housings
    .sort_values(["FIPS", "Date"])
    .groupby(["FIPS", "RegionName"])
    .agg(
        zhvi_2020=("ZHVI", "first"),
        zhvi_2025=("ZHVI", "last"),
        zori_2020=("ZORI", "first"),
        zori_2025=("ZORI", "last"),
        months=("Date", "nunique")
    ).reset_index())

# Calculate percentage growth
county_housing["zhvi_growth_pct"] = ((county_housing["zhvi_2025"] - county_housing["zhvi_2020"])
                                      / county_housing["zhvi_2020"] * 100)

county_housing["zori_growth_pct"] = ((county_housing["zori_2025"] - county_housing["zori_2020"])
                                      / county_housing["zori_2020"] * 100)

county_housing.sort_values("zhvi_growth_pct", ascending=False).reset_index(drop=True).head(10)

**Interpretation**: Santa Barbara leads at 60.5%, followed by Kern, San Diego, and Riverside, all above 50%. What stands out is how many cheaper inland counties make the top ten (Kern, Tulare, Fresno, Merced), and in those places rents grew about as fast as home values, roughly 48–52%. Santa Clara is the odd one out: home values rose 43% but rent only 16%.

**Note: this ranking only includes the 29 counties with full rent data. The full ranking of all 58 counties is in the PS3 & 4 section, where Orange County (59.8%) ranks second.**

##### RQ2: To what extent is home-value growth associated with rent growth across counties? (Only 29 Counties compared)

In [ ]:
print("correlation:", round(county_housing["zhvi_growth_pct"].corr(county_housing["zori_growth_pct"]), 2))

**Interpretation:** Across the 29 counties, home-value growth and rent growth are strongly related (correlation 0.72). Counties where home values grew faster usually also had faster rent growth. Santa Clara is the main exception: home values rose 43%, but rent rose only 16%.

#### Groupby/Agg 2: Regional housing market (58 Counties)

Next, I aggregate the county-level housing data by region. This makes it possible to compare broader geographic housing-market patterns rather than looking only at individual counties.

In [ ]:
# Group monthly housing data by county
county_housing = (
    housing
    .sort_values(["FIPS", "Date"])
    .groupby(["FIPS", "RegionName"])
    .agg(
        zhvi_2020=("ZHVI", "first"),
        zhvi_2025=("ZHVI", "last"),
        months=("Date", "nunique")
    ).reset_index())

# Calculate percentage growth
county_housing["zhvi_growth_pct"] = ((county_housing["zhvi_2025"] - county_housing["zhvi_2020"])
                                      / county_housing["zhvi_2020"] * 100)

In [ ]:
# Add the regional classification
county_housing["region"] = county_housing["RegionName"].map(county_to_region)

regional_housing = (county_housing.groupby("region").agg(
        avg_zhvi_growth=("zhvi_growth_pct", "mean"),
        counties=("FIPS", "nunique")
    ).reset_index().sort_values("avg_zhvi_growth", ascending=False)
)

regional_housing

**Interpretation**: Southern California grew the fastest (48.2% on average), followed by the Central Coast (42.0%) and the Central Valley (37.7%). The Bay Area (22.4%) and the North Coast (17.3%) grew the slowest. Some regions have only a few counties (the Central Coast has 5), so these averages should be read with caution.

#### Recode 1: Categorizing home-value growth

To make differences in housing growth easier to compare, I recode the continuous home-value growth measure into three categories.

This classification distinguishes counties with relatively low, moderate, and high home-value growth.

In [ ]:
# Use the 33rd and 67th percentiles to define relative growth categories
low_cutoff = county_housing["zhvi_growth_pct"].quantile(1/3)
high_cutoff = county_housing["zhvi_growth_pct"].quantile(2/3)

county_housing["growth_category"] = pd.cut(
    county_housing["zhvi_growth_pct"],
    bins=[-np.inf, low_cutoff, high_cutoff, np.inf],
    labels=["Low growth", "Medium growth", "High growth"]
)

county_housing["growth_category"].value_counts().sort_index()

**Interpretation**: The 33rd/67th percentile cutoffs split the 58 counties into three almost equal groups (19, 20, and 19). The groups are relative, so "Low growth" doesn't mean prices fell.

### Replace on Condition 1: Categorize unemployment conditions

I classify counties according to their unemployment rate. This allows me to compare housing growth across counties with different labor-market conditions.

In [ ]:
labor["unemployment_status"] = "Low unemployment"

labor.loc[labor["unemp_rate"] >= 5, "unemployment_status"] = "Moderate unemployment"

labor.loc[labor["unemp_rate"] >= 8, "unemployment_status"] = "High unemployment"

labor[["County", "unemp_rate", "labor_force", "unemployment_status"]].sort_values("unemp_rate", ascending=False).head()

Sierra County tops the list at 12.6%, but its labor force is only about 1,200 people, so a handful of job losses can move its rate a lot.

### Subset 1: Large California counties

I also create a subset of large counties to examine whether housing-market patterns among the largest population centers differ from the statewide county-level pattern.

In [ ]:
large_counties = labor[labor["size_tier"] == "Large (1M+)"].copy()

large_counties[["FIPS", "County", "pop_2025", "unemp_rate", "size_tier"]].sort_values("pop_2025", ascending=False)

total_pop = labor["pop_2025"].sum()
ten_counties_pop = large_counties["pop_2025"].sum()
percent = ten_counties_pop / total_pop
print(f"10 largest counties: {ten_counties_pop:,} of {total_pop:,} people ({percent:.1%})")

**Interpretation**: These ten counties hold about 72% of California's population, so these counties cover most of the people in the state. Fresno is the only one above 8% unemployment and Santa Clara the only one below 5%, so even the biggest counties face pretty different labor markets.

# ProblemSet 3 & 4


### Home-value for all 58 counties

PS2 cut `housing` to 29 counties only because of the rent data, so here I go back to all 58. Year-over-year growth needs the same month one year earlier, and `zhvi_CA` from PS1 starts in 2020, so I reload ZHVI from January 2019, compute YoY for each county, and then keep 2020–2025.

In [ ]:
zhvi_CA = zillow_to_long(zhvi, "ZHVI", "2019-01", "2026-01")

print(zhvi_CA.shape, "| counties:", zhvi_CA["FIPS"].nunique(), "| months:", zhvi_CA["Date"].nunique())

In [ ]:
home = zhvi_CA.copy()

home["region"] = home["RegionName"].map(county_to_region)
home["ym"] = home["Date"].dt.to_period("M") # month key for the merges

# year-over-year growth
home = home.sort_values(["FIPS", "Date"]).reset_index(drop=True)
home["zhvi_yoy"] = home.groupby("FIPS")["ZHVI"].pct_change(12, fill_method=None) * 100

# Keep only the original PS3 analysis period: 2020–2025.
home = home.loc[home["Date"] >= "2020-01"].reset_index(drop=True)

print(home.shape, "| counties:", home["FIPS"].nunique(), "| months:", home["ym"].nunique())
print("missing ZHVI:", home["ZHVI"].isna().sum(), "| missing YoY:", home["zhvi_yoy"].isna().sum())

# which rows have no YoY value?
home.loc[home["zhvi_yoy"].isna(), ["FIPS", "RegionName", "Date", "ZHVI", "zhvi_yoy"]]

**Interpretation**: Only four growth values are missing and all four are Mariposa. Zillow has no home value for Mariposa in September and October 2020, so those two months have no growth, and September and October 2021 have nothing one year earlier to compare with

## Mortgage Rate 30-years

30-Year Fixed Rate Mortgage Average in the United States, from Freddie Mac's weekly Primary Mortgage Market Survey.

Source: https://fred.stlouisfed.org/series/MORTGAGE30US

### Load the data

In [ ]:
import datetime
import pandas_datareader as pdr

start = datetime.datetime(2019, 1, 1)
end = datetime.datetime(2025, 12, 31)

# Freddie Mac PMMS 30-year fixed rate, weekly
mortgage_raw = pdr.DataReader('MORTGAGE30US', 'fred', start, end) # MORTGAGE30US is the code for series: know it from fed's url: https://fred.stlouisfed.org/series/MORTGAGE30US

print(mortgage_raw.shape)
print("missing:", mortgage_raw.isna().sum().sum(), "| duplicate dates:", mortgage_raw.index.duplicated().sum())

mortgage_raw.head()

In [ ]:
mortgage = (
    mortgage_raw
    .reset_index()
    .rename(columns={"DATE": "week", "MORTGAGE30US": "rate30"})
)

mortgage["week"] = pd.to_datetime(mortgage["week"])
mortgage["rate30"] = pd.to_numeric(mortgage["rate30"], errors="coerce")

print(mortgage.shape, "| missing rates:", mortgage["rate30"].isna().sum())
print("duplicate weeks:", mortgage["week"].duplicated().sum())

# PMMS is released on Thursdays, except some holiday weeks
mortgage["week"].dt.day_name().value_counts()

**Interpretation**: 353 of the 366 weekly falls on a Thursday and the other 13 on a Wednesday. Those are holiday weeks when Freddie Mac released a day early, which is one more reason not to match on the exact date.

### Weekly to monthly

Zillow is monthly, so I average the 4-5 weekly observations in each month.

I also build two variables: the 12-month change in the rate, and a recode into a low-rate vs. high-rate regime.

In [ ]:
mortgage_m = (mortgage
              .assign(mth = mortgage["week"].dt.to_period("M"))
              .groupby("mth", as_index=False)
              .agg(rate30=("rate30", "mean"),
                   n_weeks=("rate30", "count")))

# change vs. same month last year, in percentage points
mortgage_m["rate30_chg12"] = mortgage_m["rate30"] - mortgage_m["rate30"].shift(12)

# regime recode
mortgage_m["rate_regime"] = "Low rate (<5%)"
mortgage_m.loc[mortgage_m["rate30"] >= 5, "rate_regime"] = "High rate (5%+)"


print(mortgage_m.shape) # expect 84 months = 7 years x 12

# first and last month of each regime
mortgage_m.groupby("rate_regime").agg(months=("mth", "count"), first_month=("mth", "min"), last_month=("mth", "max"))

**Interpretation**: There are 84 months because I kept 2019 for the 12-month change. The rate was under 5% for 40 months in a row, from January 2019 to April 2022, and it has been at 5% or more in all 44 months since May 2022. So the recode splits the period cleanly into a before and an after, with no going back and forth.

In [ ]:
home_rate = home.merge(
    mortgage_m,
    left_on="ym",
    right_on="mth",
    how="outer",          # outer on purpose to see what doesn't match
    validate="m:1",       # many county-months to one national rate per month
    indicator=True,
)

home_rate["_merge"].value_counts()

# crosstab the merge result
pd.crosstab(home_rate["mth"].dt.year, home_rate["_merge"])

**Interpretation**: Every county-month found a rate (696 per year = 58 × 12), so nothing on the housing side is lost. The 12 `right_only` rows are the 2019 months that I kept only for the 12-month change, and the Zillow data starts in 2020, so they're expected.

In [ ]:
# keep the matched rows
home_rate = (home_rate[home_rate["_merge"] == "both"]
             .drop(columns=["_merge", "n_weeks"])
             .reset_index(drop=True))

home_rate.head(10)

### Descriptive stats: housing-cost growth vs. the mortgage rate

Median county growth for each month, with the rate on the second axis.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

state_month = (home_rate.groupby("Date", as_index=False).agg(zhvi_yoy=("zhvi_yoy", "median"),
                                                             rate30=("rate30", "first")))

fig = (make_subplots(specs=[[{"secondary_y": True}]])
       .add_trace(go.Scatter(x=state_month["Date"], y=state_month["zhvi_yoy"],
                             name="Home value YoY % (median of 58 counties)"), secondary_y=False)
       .add_trace(go.Scatter(x=state_month["Date"], y=state_month["rate30"],
                             name="30-yr mortgage rate %", line=dict(dash="dot", color="black")),
                  secondary_y=True)
       .add_hline(y=0, line_width=1, line_color="gray")
       .update_layout(title="California home-value growth vs. the 30-year mortgage rate, 2020-2025",
                      hovermode="x unified", legend=dict(orientation="h", y=-0.2))
       .update_yaxes(title_text="Year-over-year growth (%)", secondary_y=False)
       .update_yaxes(title_text="Mortgage rate (%)", secondary_y=True))

fig.show()

print("correlation between the two lines:", round(state_month["zhvi_yoy"].corr(state_month["rate30"]), 2))

**Interpretation**: Home-value growth and the mortgage rate moved in opposite directions (correlation -0.69). Growth peaked at 21.1% in August 2021, when the rate was 2.8%, and bottomed at -5.8% in July 2023, when the rate was 6.8%. The change came with a delay: the rate passed 5% in May 2022, but the typical county's year-over-year growth did not turn negative until February 2023. This is a correlation between two time series, so it shows timing, not proof of cause.

In [ ]:
# home-value growth in the low-rate vs. high-rate period
regime = (home_rate.groupby("rate_regime")
          .agg(months=("ym", "nunique"),
               avg_rate=("rate30", "mean"),
               median_yoy=("zhvi_yoy", "median"),
               share_negative=("zhvi_yoy", lambda s: (s < 0).mean() * 100))
          .round(1))
regime

**Interpretation**: In the low-rate period (January 2020 to April 2022, average rate 3.2%), the typical county's home value grew 10.9% a year, and only 1.3% of county-months had negative growth. In the high-rate period (May 2022 to December 2025, average rate 6.6%), the typical growth fell to 0.9% a year, and 40.7% of county-months were negative. So home-value growth was much weaker once the mortgage rate stayed above 5%.

## Affordability: ACS(American Community Survey) Data

Source: https://www.socialexplorer.com/data-library/2 2020-2024 (5-Year-Estimates)

### Load the data (Household Income, House Value)

In [ ]:
# Median household income at County level
household_income = pd.read_csv("https://raw.githubusercontent.com/HowardChen0211/gisPy/refs/heads/main/household_income_2024.csv", skiprows=[1])
household_income.rename(columns={"Median Household Income (In 2024 Inflation Adjusted Dollars)": "Median Household Income"}, inplace=True)
household_income.head()

In [ ]:
household_income.info()

In [ ]:
household_income.FIPS = household_income.FIPS.astype(str).str.zfill(5)

In [ ]:
# Median owner-occupied house value at County level
house_val = pd.read_csv("https://raw.githubusercontent.com/HowardChen0211/gisPy/refs/heads/main/housevalue_2024.csv", skiprows=[1])
house_val.head()

In [ ]:
house_val.info()

In [ ]:
house_val.FIPS = house_val.FIPS.astype(str).str.zfill(5)

### Clean the data

In [ ]:
census_house = (
    pop_2025[["FIPS", "County", "pop_2025"]]
    .merge(household_income[["FIPS", "Median Household Income"]], on="FIPS", how="outer", validate="one_to_one")
    .merge(house_val[["FIPS", "Median House Value"]], on="FIPS", how="outer", validate="one_to_one")
)

num_cols = ["pop_2025", "Median Household Income", "Median House Value"]
census_house[num_cols] = census_house[num_cols].apply(pd.to_numeric, errors="coerce") # to numeric types, if missing values found will fill it NaN

census_house.rename(columns={"pop_2025": "TPOP",
                     "Median Household Income": "MED_INC",
                     "Median House Value": "MED_HVAL"}, inplace = True)

In [ ]:
census_house.head()

In [ ]:
house_unemp = pd.merge(census_house, unemployment, on = ["FIPS", "County"], how = "outer", validate = "1:1", indicator = True)
house_unemp["_merge"].value_counts() # expected that 58 counties match

####  Home Price to Income Ratio

In [ ]:
house_unemp["Price2Income"] = house_unemp["MED_HVAL"] / house_unemp["MED_INC"]  # Home Price to Income Ratio

In [ ]:
# least affordable counties
(house_unemp.sort_values("Price2Income", ascending=False)
     [["County", "MED_INC", "MED_HVAL", "Price2Income"]]
     .reset_index(drop=True)
     .head(10))

In [ ]:
import plotly.express as px

fig = px.scatter(house_unemp, x="MED_INC", y="MED_HVAL",
                 color="Price2Income", hover_name="County",
                 color_continuous_scale="YlOrRd", labels={"Price2Income": "P/I Ratio"})

fig = fig.update_layout(
    title_text="Home Value vs. Household Income by County, California (ACS 2020–2024 5-Year Estimates)",
    xaxis_title="Median Household Income",
    yaxis_title="Median Home Value",
    coloraxis_colorbar=dict(title="P/I Ratio"));

fig.show()

**Interpretation:** Across California counties, home values rise faster than incomes: the price-to-income ratio climbs from about 4–6 in the lowest-income counties to roughly 10 in the highest-income ones, so the counties where people earn the most are also the least affordable.

In [ ]:
# add the region to each county
house_unemp["region"] = house_unemp["County"].map(county_to_region)


# price-to-income by region
(house_unemp.groupby("region")
     .agg(counties=("FIPS", "nunique"),
          median_income=("MED_INC", "median"),
          median_house_value=("MED_HVAL", "median"),
          median_p2i=("Price2Income", "median"))
     .reset_index()
     .sort_values("median_p2i", ascending=False))

**Interpretation:** The regions split into two clear groups. The three coastal regions all have a median ratio of 7.5 or higher (Bay Area 8.4, Central Coast 8.1, Southern California 7.5), while the three inland regions sit between 5.2 and 5.6, with almost nothing in between. The Bay Area has the highest median income, but its median house value is above $1 million, so it still ends up with the highest ratio. The Central Coast number rests on only 5 counties, so it should be read with caution.

## Unemployment Rate vs. Home-Price Growth

#### Merge: ZHVI x Unemployment

In [ ]:
# monthly ZHVI, 2020-01 to 2025-12
county_zhvi = (
    home
    .sort_values(["FIPS", "Date"])
    .groupby(["FIPS", "RegionName", "region"])
    .agg(zhvi_2020=("ZHVI", "first"),    # Jan 2020
         zhvi_2025=("ZHVI", "last"),     # Dec 2025
         months=("Date", "nunique"))
    .reset_index())

county_zhvi["zhvi_growth_pct"] = (county_zhvi["zhvi_2025"] / county_zhvi["zhvi_2020"] - 1) * 100

county_zhvi.sort_values("zhvi_growth_pct", ascending=False).reset_index(drop=True).head()

In [ ]:
growth = county_zhvi.merge(
    unemployment.drop(columns="County"),
    on="FIPS",
    how="outer",
    validate="1:1",
    indicator=True,
)

growth["_merge"].value_counts() # expected.

#### Scatter Plot

In [ ]:
df = growth.dropna(subset=["unemp_rate", "zhvi_2025", "zhvi_growth_pct"]).copy()
df["log_zhvi_2025"] = np.log(df["zhvi_2025"])

for col in ["zhvi_2025", "log_zhvi_2025", "zhvi_growth_pct"]:
    p = df["unemp_rate"].corr(df[col])
    print(f"{col:16s} pearson={p:.3f}")

fig = px.scatter(
    df, x="unemp_rate", y="zhvi_2025", color="region",
    hover_name="RegionName", log_y=True,
    trendline="ols", trendline_scope="overall",
    trendline_options=dict(log_y=True),
    title="Unemployment Rate vs. Home Value (Dec 2025), California counties",
    labels={"unemp_rate": "Unemployment Rate (%)",
            "zhvi_2025": "ZHVI, Dec 2025 (log scale)"},
)
fig.show()

**Interpretation**: Counties with higher unemployment have cheaper homes, the correlation between the unemployment rate and the December 2025 home value is -0.51 (-0.56 on a log scale). But unemployment is not related to how fast home values grew (correlation 0.02). So the local job market helps explain which counties are expensive, not which ones grew the fastest.

## Population Growth vs. Home-Price Growth

Source: Census Bureau; Social Explorer. Total Population, 2020. Prepared by Social Explorer. https://www.socialexplorer.com/data/CDHC2020/metadata?ds=SE&table=T001

In [ ]:
pop_2020 = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/TPOP_2020.csv")
pop_2020.head()

In [ ]:
pop_2020.info()

### Clean the data

In [ ]:
pop_2020["Geo_FIPS"] = pop_2020["Geo_FIPS"].astype(str).str.zfill(5)

pop_2020.rename(columns={"Geo_FIPS":"FIPS", "Geo_NAME":"County", "SE_T001_001": "pop_2020"}, inplace=True)

pop_2020 = pop_2020[["FIPS", "County", "pop_2020"]]

pop_2020.head(10)

### Merge: Population 2020 x Population 2025

In [ ]:
pop_growth = pop_2020.merge(
    pop_2025.drop(columns="County"),
    on="FIPS",
    how="outer",          # outer on purpose, to see what doesn't match
    validate="1:1",
    indicator=True,
)

pop_growth["_merge"].value_counts()

#### To See the Population Growth in 5 years (2020 ~ 2025)

In [ ]:
pop_growth = pop_growth.drop(columns="_merge")

# population change, in people and in percent
pop_growth["pop_change"] = pop_growth["pop_2025"] - pop_growth["pop_2020"]
pop_growth["pop_growth_pct"] = (pop_growth["pop_2025"] / pop_growth["pop_2020"] - 1) * 100

# statewide total
ca_2020, ca_2025 = pop_growth["pop_2020"].sum(), pop_growth["pop_2025"].sum()
print(f"California: {ca_2020:,} -> {ca_2025:,} ({(ca_2025 / ca_2020 - 1) * 100:.2f}%)\n")
print("Counties that lost population:", (pop_growth["pop_growth_pct"] < 0).sum(), "of", len(pop_growth))

# five fastest-growing and five fastest-shrinking counties
pop_sorted = pop_growth.sort_values("pop_growth_pct", ascending=False).reset_index(drop=True)
pop_sorted.head()
print()
pop_sorted.tail()

**Interpretation:** California lost about 0.5% of its population between 2020 and 2025, and 37 of the 58 counties shrank. Placer, San Benito, and Yuba grew the fastest (about 9%), while Lassen and Alpine lost more than 13%. Note that the 2020 number is the Census count and the 2025 number is an estimate, so small changes should be read with caution.

#### To See the Housing Values Growth in 5 years (2020 ~ 2025)

In [ ]:
# monthly ZHVI, 2020-01 to 2025-12
county_zhvi = (
    home
    .sort_values(["FIPS", "Date"])
    .groupby(["FIPS", "RegionName", "region"])
    .agg(zhvi_2020=("ZHVI", "first"),    # Jan 2020
         zhvi_2025=("ZHVI", "last"),     # Dec 2025
         months=("Date", "nunique"))
    .reset_index())

county_zhvi["zhvi_growth_pct"] = (county_zhvi["zhvi_2025"] / county_zhvi["zhvi_2020"] - 1) * 100

county_zhvi.sort_values("zhvi_growth_pct", ascending=False).reset_index(drop=True).head()

In [ ]:
print("median growth:", round(county_zhvi["zhvi_growth_pct"].median(), 1))

print("counties with higher home values:", (county_zhvi["zhvi_growth_pct"] > 0).sum(), "of", len(county_zhvi))

county_zhvi.sort_values("zhvi_growth_pct").head(3)[["RegionName", "zhvi_growth_pct"]]

**Interpretation:** With all 58 counties, Santa Barbara (60.5%) and Orange (59.8%) lead, and Southern California and the Central Valley take eight of the top ten. The median county grew 31.9%, and only Mendocino (-4.8%), Trinity (-3.1%), and San Francisco (-2.4%) ended lower than they started.

### Merge: ZHVI x Population Growth

In [ ]:
growth = county_zhvi.merge(
    pop_growth.drop(columns="County"),
    on="FIPS",
    how="outer",
    validate="1:1",
    indicator=True,
)

growth["_merge"].value_counts() # expected.

### Scatter Plot

In [ ]:
growth = growth.drop(columns="_merge")

print("correlation:", round(growth["pop_growth_pct"].corr(growth["zhvi_growth_pct"]), 3))

fig = px.scatter(growth, x="pop_growth_pct", y="zhvi_growth_pct",
                 hover_name="RegionName", trendline="ols",
                 title="Population Growth vs. home-value Growth, California counties, 2020-2025",
                 labels={"pop_growth_pct": "Population Growth (%)",
                         "zhvi_growth_pct": "Home-value Growth (%)"})
fig.show()

**Interpretation:** The relationship is positive but weak (correlation 0.26). Counties that gained people saw slightly faster home-value growth, but population change explains only about 7% of the differences between counties (R² = 0.07). Because this is a correlation, it does not show whether population change causes price changes.

In [ ]:
# county size vs. home-value growth
size = county_zhvi.merge(labor[["FIPS", "size_tier", "pop_2025"]],
                         on="FIPS", how="left", validate="1:1")

print("Correlation: log population vs. growth",
      round(np.log(size["pop_2025"]).corr(size["zhvi_growth_pct"]), 3))

(size.groupby("size_tier", observed=True)["zhvi_growth_pct"]
     .agg(counties="count", median_growth="median")
     .round(1))

**Interpretation**: Small counties grew a median of 28.6%, mid-sized counties 32.3%, and large counties 44.2%. The correlation between county size (log population) and growth is 0.271, so size explains only a small part of the differences.

## Reference

1. Zillow Home Value Index (ZHVI), county, all homes, mid tier (35th–65th pct), smoothed, seasonally adjusted, monthly: https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv
   (Zillow Research data page: https://www.zillow.com/research/data/  *Home Values* Data Type "ZHVI All Homes (SFR, Condo/Co-op) Time Series, Smoothed, Seasonally Adjusted", Geography "County")

2. Zillow Observed Rent Index (ZORI), county, all homes plus multifamily, smoothed, monthly: https://files.zillowstatic.com/research/public_csvs/zori/County_zori_uc_sfrcondomfr_sm_month.csv

3. U.S. Census Bureau, American Community Survey 2020-2024 5-Year Estimates, table A17005 (Employment Status for Total Population 16 Years and Over), CA counties, via Social Explorer: https://www.socialexplorer.com/reports/socialexplorer/en/report/bac15980-b308-11f1-9512-c38da70940e6

4. California Air Resources Board, Cool California regions (county-to-region grouping): https://coolcalifornia.arb.ca.gov/regions

5. Freddie Mac, 30-Year Fixed Rate Mortgage Average in the United States [MORTGAGE30US], Primary Mortgage Market Survey, weekly, retrieved from FRED, Federal Reserve Bank of St. Louis: https://fred.stlouisfed.org/series/MORTGAGE30US

6. Census Bureau; Social Explorer. Total Population, 2020. Prepared by Social Explorer. https://www.socialexplorer.com/data/CDHC2020/metadata?ds=SE&table=T001 (accessed 5 October 2026 at 15:17:18 GMT-4).

7. Social Explorer; U.S. Census Bureau. Median Household Income (A14006), ACS 2024 5-Year. https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A14006

8. Social Explorer; U.S. Census Bureau. Median House Value (A10036), ACS 2024 5-Year. https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A10036

9. Claude https://claude.ai/